# 🎬 Netflix Global Content & Expansion Strategy Analysis

---

### 🎯 Selected Project Challenge
**Challenge 2: Global Expansion Strategy**  
> **Core Business Question:** *Which countries contribute the most content to Netflix, and where could Netflix expand next?*

### 📋 Project Overview & Objectives
In an increasingly competitive global streaming landscape, understanding geographic content supply, localization gaps, and regional catalog balance is critical for subscriber growth. Using a realistic, noisy 5,000-record Netflix dataset (`netflix_global_expansion_raw_5000.csv`), our team executes an end-to-end data analytics workflow—from raw data remediation and feature extraction to exploratory analysis, executive visualizations, and actionable expansion recommendations.

---

## Project Team

| No. | Team Member | Role |
|---:|---|---|
| 1 | Ashutosh Shahi | Data Cleaning |
| 2 | Aryan Sutariya | EDA |
| 3 | Mahadev Mishra | Visualization |
| 4 | Monalisa Raut | Final Report |

---

# **-------------Part-1 : Ashutosh Shahi --------------**

### 🛠️ Data Quality Audit & Remediation Plan
Before analyzing global expansion opportunities, the raw 5,000-row dataset (`netflix_global_expansion_raw_5000.csv`) requires systematic cleaning across several data quality issues:
1. **Duplicate & Near-Duplicate Records:** Exact duplicates and case-shifted title duplicates inflate country counts.
2. **Inconsistent Categorical Labels:** `type` contains variants (`Movie`, `movie`, `MOVIE`, `TV-Show`, `Tv show`, `'nan'`); `country` contains aliases (`USA`, `US`, `U.S.A.`, `UK`, `U.K.`, `Great Britain`, `Korea, South`, `Deutschland`, `Brasil`) and mixed delimiters (`,`, `/`, `;`).
3. **Misplaced & Corrupted Values:** Duration strings (`74 min`, `84 min`) appear inside `rating`; invalid years (`-1`, `0`, `1925`, `2099`) appear in `release_year`; HTML tags (`<p>...</p>`) and trailing IDs pollute `description`.
4. **Unstructured Metadata Extraction:** The `unstructured_metadata` column stores heterogeneous strings (`Views~... | IMDb:...`, `region_tier=...; score=...; lang=...`) that must be parsed via regular expressions into structured numeric and categorical features (`estimated_views`, `imdb_score`, `region_tier`).

In [4]:
import pandas as pd
import numpy as np
import re
import warnings

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)

raw_file_path = '../data/netflix_global_expansion_raw_5000.csv'
df_raw = pd.read_csv(raw_file_path)

print(f"Raw Dataset Shape: {df_raw.shape[0]:,} rows x {df_raw.shape[1]} columns")
print("\nMissing Values Before Cleaning:")
print(df_raw.isna().sum()[df_raw.isna().sum() > 0])
df_raw.head(5)

Raw Dataset Shape: 5,000 rows x 13 columns

Missing Values Before Cleaning:
type                       49
title                      17
director                 1403
cast                      461
country                   390
date_added                 62
rating                    190
duration                   47
listed_in                  44
description               109
unstructured_metadata    1015
dtype: int64


,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description,unstructured_metadata
0,s1502,Movie,Silent Madrid 3,"Vikram Kowalski, Mei Novak",Mei Hassan | Liam Nair | Omar Takahashi | Davi...,Indonesia,"September 12, 2025",1991,PG-13,103 min,Horror Movies / Stand-Up Comedy,An intimate look into the vibrant culture and ...,"Views~39,632,495 | IMDb:8.2/10 | Subtitles:EN,..."
1,s2587,Movie,Frozen Paris 6,NaN,"Fatima Khan, Sofia Kim, Rahul Takahashi, Isabe...",NaN,"Dec 24, 2025",2012,TV-PG,80 min,Horror Movies / Comedies,"Set in United Kingdom, a determined architect ...",N/A - pending audit
2,s2654,Movie,Crimson Lagos 6,Claire Chen,Omar Kowalski | Nadia Brown | Jin Kim | Yuki S...,South Korea,"Sep 29, 2019",2017,PG,123 min,"action & adventure, Sports Movies, Independent...",NaN,N/A - pending audit
3,s1056,Movie,Rise of Dreamers 5,NaN,Chloe Hassan; Marcus Nair; Gabriel O'Connor; L...,Poland,"July 31, 2023",2016,TV-14,89 min,"Dramas, Horror Movies","When a sudden crisis strikes Poland, an unlike...",region_tier=Tier-1; score=6.1; lang=English
4,s706,Movie,Rise of Berlin,Noah Dubois,"Claire Kowalski, Gabriel Laurent, Rahul Nair, ...",Thailand,"March 03, 2025",2015,TV-MA,120 min,"Music & Musicals, Stand-Up Comedy","A heartwarming comedy about family, tradition,...",N/A - pending audit


In [5]:
df_clean = df_raw.copy()

df_clean['show_id'] = df_clean['show_id'].astype(str).str.strip().str.lower().str.replace('-', '', regex=False)
df_clean['title'] = (
    df_clean['title']
    .astype(str)
    .replace({'nan': np.nan, 'None': np.nan, '': np.nan})
    .str.strip()
    .str.replace(r'\s*###$', '', regex=True)
)
df_clean = df_clean.dropna(subset=['title']).copy()
df_clean['title_norm'] = df_clean['title'].str.lower().str.strip()
df_clean = df_clean.drop_duplicates(subset=['title_norm'], keep='first').drop(columns=['title_norm'])

def clean_content_type(val):
    if pd.isna(val) or str(val).strip().lower() in ['nan', 'none', '']:
        return np.nan
    s = str(val).strip().lower().replace('-', ' ')
    if 'movie' in s:
        return 'Movie'
    if 'tv' in s or 'show' in s:
        return 'TV Show'
    return np.nan

df_clean['type'] = df_clean['type'].apply(clean_content_type)
mask_missing_type = df_clean['type'].isna()
df_clean.loc[mask_missing_type & df_clean['duration'].astype(str).str.contains('Season', case=False, na=False), 'type'] = 'TV Show'
df_clean['type'] = df_clean['type'].fillna('Movie')

country_alias_map = {
    'usa': 'United States', 'us': 'United States', 'u.s.a.': 'United States', 'united states': 'United States',
    'uk': 'United Kingdom', 'u.k.': 'United Kingdom', 'united kingdom': 'United Kingdom', 'great britain': 'United Kingdom',
    'korea, south': 'South Korea', 's. korea': 'South Korea', 'south korea': 'South Korea',
    'india': 'India', 'can': 'Canada', 'canada': 'Canada',
    'jpn': 'Japan', 'japan': 'Japan',
    'deutschland': 'Germany', 'germany': 'Germany',
    'fra': 'France', 'france': 'France',
    'esp': 'Spain', 'spain': 'Spain',
    'brasil': 'Brazil', 'brazil': 'Brazil',
    'nga': 'Nigeria', 'nigeria': 'Nigeria'
}

def standardize_country_list(val):
    if pd.isna(val):
        return 'Unknown'
    raw_str = str(val).strip()
    if raw_str.lower() in ['nan', 'n/a', 'unknown', '---', '']:
        return 'Unknown'
    parts = re.split(r'[,/;]+', raw_str)
    cleaned_parts = []
    for p in parts:
        token = p.strip()
        if not token or token.lower() in ['nan', 'n/a', 'unknown', '---']:
            continue
        mapped = country_alias_map.get(token.lower(), token.title())
        if mapped not in cleaned_parts:
            cleaned_parts.append(mapped)
    return ', '.join(cleaned_parts) if cleaned_parts else 'Unknown'

df_clean['country'] = df_clean['country'].apply(standardize_country_list)
df_clean['primary_country'] = df_clean['country'].apply(lambda x: x.split(',')[0].strip())
df_clean['is_coproduction'] = df_clean['country'].apply(lambda x: 1 if ',' in x else 0)

valid_ratings = {'TV-MA', 'TV-14', 'TV-PG', 'R', 'PG-13', 'TV-Y7', 'TV-Y', 'PG', 'TV-G', 'NR', 'G', 'UR'}
def clean_rating(val):
    if pd.isna(val) or str(val).strip().lower() in ['nan', 'none', '']:
        return 'Not Rated'
    s = str(val).strip().upper()
    if s == 'TV14':
        s = 'TV-14'
    if 'MIN' in s or s not in valid_ratings:
        return 'Not Rated'
    return s

df_clean['rating'] = df_clean['rating'].apply(clean_rating)

median_year = int(df_clean.loc[df_clean['release_year'].between(1980, 2026), 'release_year'].median())
df_clean['release_year'] = df_clean['release_year'].apply(lambda y: y if 1980 <= y <= 2026 else median_year)

df_clean['date_added_clean'] = pd.to_datetime(df_clean['date_added'].astype(str).str.strip(), errors='coerce')
df_clean['year_added'] = df_clean['date_added_clean'].dt.year.fillna(df_clean['release_year']).astype(int)

def extract_duration_num(val):
    if pd.isna(val):
        return np.nan
    m = re.search(r'(\d+)', str(val))
    return float(m.group(1)) if m else np.nan

df_clean['duration_value'] = df_clean['duration'].apply(extract_duration_num)
df_clean['duration_value'] = df_clean.groupby('type')['duration_value'].transform(lambda s: s.fillna(s.median()))

df_clean['director'] = df_clean['director'].fillna('Not Specified').astype(str).str.strip()
df_clean['cast'] = df_clean['cast'].fillna('Not Specified').astype(str).str.strip()
df_clean['listed_in'] = df_clean['listed_in'].fillna('General Entertainment').astype(str).str.strip()
df_clean['description'] = (
    df_clean['description']
    .fillna('No description available.')
    .astype(str)
    .str.replace(r'<[^>]+>', '', regex=True)
    .str.replace(r'\[ID:\d+\]', '', regex=True)
    .str.strip()
)

def parse_metadata(text):
    if pd.isna(text) or 'pending audit' in str(text).lower():
        return pd.Series([np.nan, np.nan, 'Unspecified'])
    s = str(text)
    imdb_match = re.search(r'(?:IMDb[:\s]*|score=)(\d+\.\d+)', s)
    imdb_val = float(imdb_match.group(1)) if imdb_match else np.nan
    views_match = re.search(r'(?:Views~|\()([\d,]+)', s)
    views_val = float(views_match.group(1).replace(',', '')) if views_match else np.nan
    tier_match = re.search(r'region_tier=([A-Za-z0-9\-]+)', s)
    tier_val = tier_match.group(1) if tier_match else 'Unspecified'
    return pd.Series([imdb_val, views_val, tier_val])

df_clean[['imdb_score', 'estimated_views', 'region_tier']] = df_clean['unstructured_metadata'].apply(parse_metadata)
df_clean['imdb_score'] = df_clean['imdb_score'].fillna(df_clean['imdb_score'].median())
df_clean['estimated_views'] = df_clean['estimated_views'].fillna(df_clean['estimated_views'].median())

print(f"Cleaned Dataset Shape: {df_clean.shape[0]:,} rows x {df_clean.shape[1]} columns")
print(f"Duplicates & Invalid Titles Removed: {len(df_raw) - len(df_clean):,}")
print(f"Remaining Null Values Across Cleaned Columns: {df_clean.drop(columns=['date_added', 'duration', 'unstructured_metadata', 'date_added_clean']).isna().sum().sum()}")
df_clean[['show_id', 'type', 'title', 'primary_country', 'is_coproduction', 'release_year', 'rating', 'imdb_score', 'estimated_views']].head(5)

Cleaned Dataset Shape: 4,109 rows x 21 columns
Duplicates & Invalid Titles Removed: 891
Remaining Null Values Across Cleaned Columns: 0


,show_id,type,title,primary_country,is_coproduction,release_year,rating,imdb_score,estimated_views
0,s1502,Movie,Silent Madrid 3,Indonesia,0,1991,PG-13,8.20,39632495.0
1,s2587,Movie,Frozen Paris 6,Unknown,0,2012,TV-PG,6.65,41829086.0
2,s2654,Movie,Crimson Lagos 6,South Korea,0,2017,PG,6.65,41829086.0
3,s1056,Movie,Rise of Dreamers 5,Poland,0,2016,TV-14,6.10,41829086.0
4,s706,Movie,Rise of Berlin,Thailand,0,2015,TV-MA,6.65,41829086.0
